In [ ]:
import os
import glob
import numpy as np
import pandas as pd

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.common.qc import run_session_synapse_qc
from vip_slap2_analysis.behavior.preprocess import process_behavior_session

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
target_mice = [
    #803496,804730,804733,810196,809047,803121,
               826033]

In [ ]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check","volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [ ]:
meta = {
    "epochs_mode": "continuous",
    "prepost_sec": {
        "image_identity": (0.25, 0.50),
        "change": (1.00, 0.75),
        "omission": (1.00, 1.50),
    },
}

In [ ]:
for asset in assets:
    print(f'Initiating QC and processing for {asset.session_id}')
    print("summary:", asset.summary_mat)
    print("bonsai:", asset.bonsai_event_log_csv)
    print("harp:", asset.harp_dir)
#     print("photodiode:", asset.photodiode_pkl)
    print('')
    asset.ensure_dirs()
    
    print(f"Processing behavior data...")
    
    behavior_result = process_behavior_session(
        asset,
        metadata=meta,
        overwrite_harp_extract=False,
        overwrite_alignment=False,
        save_corrected_in_place=True,
        min_epoch_duration=6.0,
        trial_gap_start=0.02,
        expected_trial_epoch_min=None,
        use_piecewise_warp=True,
    )

    print(behavior_result.status)
    print(behavior_result.failure_stage)
    print(behavior_result.failure_reasons)

    if not behavior_result.ready_for_physiology_extraction:
        print(f"Skipping physiology extraction for {asset.session_id}")
        print('\n')
        continue
    
    print(f"\nProcessing physiology data...")
    
    physiology_result = run_session_synapse_qc(
    summary_mat_path=asset.summary_mat,
    output_dir=asset.qc_dir/"slap2",
    session_id=asset.session_id,
    subject_id=asset.subject_id,
    trace_signal='dF',
    trace_mode='ls',
    save=True,
    make_plots=True)
    
    print(f'Finished running QC on {asset.session_id}\n')